# Medical Assistant LoRA Fine-tuning

Fine-tune TinyLlama-1.1B-Chat on medical symptom inquiry dataset using QLoRA.

In [1]:
#=================================
#  Install Dependencies (Run once)
#=================================

!pip install -q transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.6 MB/s eta 0:00:00


In [2]:
#==================
# Imports and Setup
#==================

import torch
import re
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# Verify installation
import bitsandbytes as bnb
print(f"✅ bitsandbytes: {bnb.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ CUDA: {torch.version.cuda}")

✅ bitsandbytes: 0.49.2
✅ GPU: Tesla T4
✅ CUDA: 12.8


In [57]:
#=======================
# Load and Parse Dataset
#=======================

def parse_medical_dataset(file_path):
    """Parse the medical dataset into clean conversation format."""
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Split by the main separator (==================================================)
    entries = content.split("==================================================")

    conversations = []
    for entry in entries:
        entry = entry.strip()
        if not entry:
            continue

        # Skip metadata lines (ID, INTENT, DIFFICULTY) and the secondary separator
        lines = entry.split("\n")
        conversation_lines = []
        skip_until_content = False

        for line in lines:
            line = line.strip()
            # Skip metadata and separator lines
            if line.startswith("ID:") or line.startswith("INTENT:") or line.startswith("DIFFICULTY:"):
                continue
            if line.startswith("---"):  # Skip the "------------------------------" separator
                skip_until_content = True
                continue
            if skip_until_content:
                conversation_lines.append(line)

        # Join conversation lines and clean up
        conversation_text = "\n".join(conversation_lines).strip()

        # Remove image references like [Image of ...]
        conversation_text = re.sub(r'\[Image of[^\]]*\]', '', conversation_text)

        # Remove extra blank lines
        conversation_text = re.sub(r'\n{3,}', '\n\n', conversation_text)

        if conversation_text and len(conversation_text) > 50:  # Filter out empty/short entries
            conversations.append(conversation_text)

    return conversations

# Parse the dataset
conversations = parse_medical_dataset("/medical_assistant_dataset.txt")
print(f"Loaded {len(conversations)} conversations")

# Print first conversation to verify
print("\n Sample conversation:")
print("-" * 50)
print(conversations[0][:500] if conversations else "No conversations found")
print("-" * 50)

Loaded 30000 conversations

 Sample conversation:
--------------------------------------------------
User: I've been experiencing bradycardia lately. What is that exactly?
Assistant: I understand your concern about these symptoms. a heart rate that is slower than the typical range, which can sometimes result in fatigue or lightheadedness.

Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice. I strongly recommend consulting a licensed healthcare professional for a personalized evaluation.
User: Is there anything else I should watch out for?
Assistant: Have yo
--------------------------------------------------


In [30]:
#====================
# Model Configuration
#====================

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Using: {model_name}")

Using: Qwen/Qwen2.5-0.5B-Instruct


In [31]:
#==========================
#  Load Model and Tokenizer
#==========================

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Prepare for Training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Loaded {model_name}")
print(f"✅ Model size: ~500M parameters")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded Qwen/Qwen2.5-0.5B-Instruct
✅ Model size: ~500M parameters


In [58]:
#=====================
# Format Conversations
#=====================

def format_chat(conversation_text):
    """Universal chat format."""
    # Parse User: and Assistant: turns
    turns = []
    lines = conversation_text.split("\n")
    current_role = None
    current_content = []

    for line in lines:
        line = line.strip()
        if line.startswith("User:"):
            if current_role and current_content:
                turns.append({"role": current_role, "content": " ".join(current_content).strip()})
            current_role = "user"
            current_content = [line[5:].strip()]
        elif line.startswith("Assistant:"):
            if current_role and current_content:
                turns.append({"role": current_role, "content": " ".join(current_content).strip()})
            current_role = "assistant"
            current_content = [line[10:].strip()]
        elif line:
            current_content.append(line)

    if current_role and current_content:
        turns.append({"role": current_role, "content": " ".join(current_content).strip()})

    # Simple format
    chat_text = ""
    for turn in turns:
        if turn["role"] == "user":
            chat_text += f"User: {turn['content']}\n"
        else:
            chat_text += f"Assistant: {turn['content']}\n"

    return chat_text.strip()

# Format conversations
formatted_conversations = [format_chat(conv) for conv in conversations]
dataset = Dataset.from_list([{"text": text} for text in formatted_conversations])

print(f" Dataset ready: {len(dataset)} examples")

 Dataset ready: 30000 examples


In [33]:
#=================
# Tokenize Dataset
#=================

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"✅ Tokenized {len(tokenized_dataset)} examples")

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

✅ Tokenized 30000 examples


In [34]:
#===================
# Lora Configuration
#===================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [35]:
#===============
# Data Collector
#===============

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("✅ Model ready for training")

✅ Model ready for training


In [36]:
#===================
# Training Arguments
#===================

training_args = TrainingArguments(
    output_dir="./medical_lora_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=5e-4,
    max_grad_norm=0.3,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

In [37]:
#====================
# Initiallize Trainer
#====================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("✅ Trainer initialized")

✅ Trainer initialized


In [38]:
#=========
# Training
#=========

print("🚀 Starting training...")
trainer.train()

🚀 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.420054
10,1.936185
15,1.706130
20,1.994072
25,0.939198
30,0.848331
35,0.530616
40,0.436457
45,0.549832
50,0.360592


TrainOutput(global_step=7500, training_loss=0.04907780856291453, metrics={'train_runtime': 3207.7489, 'train_samples_per_second': 9.352, 'train_steps_per_second': 2.338, 'total_flos': 8270886666240000.0, 'train_loss': 0.04907780856291453, 'epoch': 1.0})

In [59]:
#================
# Test Generation
#================

model.eval()

# Use ChatML format for inference
prompt = """<|user|>
I've been experiencing bradycardia lately. What is that exactly?</s>
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.convert_tokens_to_ids("</s>"),
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(f"\n Response:\n{response}")


 Response:
<|user|>
I've been experiencing bradycardia lately. What is that exactly?</s>
<|assistant|>
I understand your concern about these symptoms. a heart rate that is slower than the typical range, which can sometimes result in fatigue or lightheadedness. Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice. I strongly recommend consulting a licensed healthcare professional for a personalized evaluation.
<|name|>
Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice. I strongly recommend consulting a licensed healthcare professional for a personalized evaluation. Please feel free to question any of these statements. Is there anything else you would like to discuss? Please remember that I am an AI, not a doctor. This is not a diagnosis or medical advice. I strongly recommend consulting
